# Art Style Classification on WikiArt

`Runtime → Change runtime type → T4 GPU`

Mount Google Drive (Cell 1)

## 0. Setup

In [ ]:
# Mount Google Drive (models and data will be saved here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# All outputs go to Google Drive so they survive session resets
DRIVE_ROOT   = '/content/drive/MyDrive/wikiart'
CKPT_DIR     = os.path.join(DRIVE_ROOT, 'checkpoints')
OUTPUT_DIR   = os.path.join(DRIVE_ROOT, 'outputs')
DATA_RAW     = os.path.join(DRIVE_ROOT, 'data/raw')
DATA_FULL    = os.path.join(DRIVE_ROOT, 'data/processed')

for d in [CKPT_DIR, OUTPUT_DIR, DATA_RAW, DATA_FULL]:
    os.makedirs(d, exist_ok=True)

print('Drive folders ready.')

In [ ]:
# Clone your GitHub repo
# Replace with your actual repo URL after pushing
REPO_URL = 'https://github.com/YOUR_USERNAME/wikiart-style-classification'

if not os.path.exists('/content/wikiart'):
    !git clone {REPO_URL} /content/wikiart
else:
    !cd /content/wikiart && git pull

%cd /content/wikiart

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
# Verify GPU is available
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 1. Prepare Dataset
Downloads WikiArt and creates balanced train/val/test splits (1500 images/class).  
**Skip this cell if `data/processed` already exists in Drive.**

In [ ]:
# Check if dataset already prepared
if os.path.exists(os.path.join(DATA_FULL, 'train')):
    print('Dataset already prepared, skipping download.')
else:
    !python prepare_dataset.py \
        --source_dir {DATA_RAW} \
        --output_dir {DATA_FULL}

## 2. Train CNN Baseline

In [ ]:
!python train_baseline.py \
    --data_dir   {DATA_FULL} \
    --epochs     20 \
    --batch_size 32 \
    --lr         1e-3

In [ ]:
# Copy checkpoint to Drive
!cp checkpoints/cnn_baseline.pth {CKPT_DIR}/
!cp outputs/cnn_baseline_curves.png {OUTPUT_DIR}/
print('Saved to Drive.')

## 3. Transfer Learning — ResNet18 (frozen backbone)

In [ ]:
!python train_transfer.py \
    --data_dir   {DATA_FULL} \
    --strategy   frozen \
    --epochs     20 \
    --batch_size 32

In [ ]:
!cp checkpoints/resnet18_frozen.pth {CKPT_DIR}/
!cp outputs/resnet18_frozen_curves.png {OUTPUT_DIR}/
print('Saved to Drive.')

## 4. Transfer Learning — ResNet18 (full fine-tuning)

In [ ]:
!python train_transfer.py \
    --data_dir   {DATA_FULL} \
    --strategy   full \
    --epochs     20 \
    --batch_size 32

In [ ]:
!cp checkpoints/resnet18_full.pth {CKPT_DIR}/
!cp outputs/resnet18_full_curves.png {OUTPUT_DIR}/
print('Saved to Drive.')

## 5. Evaluate All Models

In [ ]:
!python evaluate.py --data_dir {DATA_FULL}

In [ ]:
# Copy confusion matrices to Drive
!cp outputs/confusion_*.png {OUTPUT_DIR}/
print('Evaluation outputs saved to Drive.')

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['CNN Baseline', 'ResNet18 (frozen)', 'ResNet18 (full)']
files  = ['confusion_CNN_Baseline.png',
          'confusion_ResNet18_frozen.png',
          'confusion_ResNet18_full.png']

for ax, title, fname in zip(axes, titles, files):
    path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title)
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'all_confusion_matrices.png'), dpi=150)
plt.show()

In [ ]:
# Training curves side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
curve_files = ['cnn_baseline_curves.png',
               'resnet18_frozen_curves.png',
               'resnet18_full_curves.png']

for ax, title, fname in zip(axes, titles, curve_files):
    path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
        ax.set_title(title)
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'all_training_curves.png'), dpi=150)
plt.show()

## 7. Push Best Model to Hugging Face Hub
Run this after training to upload the model for the Gradio demo.

In [ ]:
# Install huggingface_hub if needed
!pip install huggingface_hub -q

from huggingface_hub import HfApi

# Replace with your HF token and repo name
HF_TOKEN   = 'hf_YOUR_TOKEN_HERE'
HF_REPO_ID = 'YOUR_HF_USERNAME/wikiart-style-classifier'

api = HfApi()

# Create repo if it doesn't exist
api.create_repo(repo_id=HF_REPO_ID, token=HF_TOKEN, exist_ok=True)

# Upload best checkpoint
api.upload_file(
    path_or_fileobj=os.path.join(CKPT_DIR, 'resnet18_full.pth'),
    path_in_repo='resnet18_full.pth',
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)
print(f'Model uploaded to https://huggingface.co/{HF_REPO_ID}')